
### Parameters

In [ ]:
dictParameters = {
  "thresholdMissingValues": {
    "value": 20,
    "description": "Percentage threshold used to identify features with missing values. Features with less than 20% missing values will have their missing values filled/imputed."
  },
  "thresholdNaValues": {
    "value": 40,
    "description": "Percentage threshold used to identify features with a high proportion of NUll Values, if a feature exceeds that threshold we are asussming that the feature allows NA values"
  },
  "thresholdCollinearity": {
    "value": 70,
    "description": "Percentage threshold used to identify highly collinear features. Features with a collinearity greater than 70% can be considered redundant, the feature that has less correlation will be removed."
  },
  "thresholdCorrelation": {
    "value": 5,
    "description": "Threshold used to determine which features have a relevant correlation with the target to consider."
  },
  "testSplitSize": {
    "value": 20,
    "description": "Percentage of the dataset reserved for testing. In this case, 20% of the data is used as the test set."
  },
  "thresholdStandarization": {
    "value": 10,
    "description": "Minimum number of unique values required for a numerical feature to be considered for standardization. by default features with fewer than 10 unique values can be excluded from standardization because we are considering those feature as rating features."
  },
  "thresholdOneHotEncoding": {
    "value": 5,
    "description": "Maximum number of unique categorical values allowed for a feature to be considered for One-Hot Encoding, otherwise we will use TargetEncoding."
  },
  "iterationOffset": {
    "value": 2,
    "description": "Number of columns excluded from the beginning of the iteration range to evaluate the best combination feature. For example, with 10 columns and an offset of 2, the iterations will use 9 and 10 columns."
  }
}

In [ ]:
thresholdMissingValues = dictParameters["thresholdMissingValues"]["value"]
ThresHoldNaValues = dictParameters["thresholdNaValues"]["value"]
ThresholdCollinearity = dictParameters["thresholdCollinearity"]["value"]
ThresholdCorrelation = dictParameters["thresholdCorrelation"]["value"]
testSplitSize = dictParameters["testSplitSize"]["value"]
thresholdOneHotEncoding = dictParameters["thresholdOneHotEncoding"]["value"]
iteration_offset = dictParameters["iterationOffset"]["value"]


ThresHoldStandarizacion  = dictParameters["thresholdStandarization"]["value"]

## reading_source

In [ ]:
import pandas as pd
pd.set_option('display.max_rows', 500)
from tabulate import tabulate

dfSummaryFillNaValues = []
dfSummaryDataStandarizacion = []
url = "https://marksmath.org/data/house-prices/train.csv"

house_data = pd.read_csv(url)


dfDeletedColumns = pd.DataFrame( columns=  ['column_name','type',  'status', 'reason'])

def detectValueType(column):
        valueType = house_data[column].dtype.type

        if pd.api.types.is_integer_dtype(valueType):
            feature_type = 'int'
        elif pd.api.types.is_float_dtype(valueType):
            feature_type = 'float'
        elif pd.api.types.is_bool_dtype(valueType):
            feature_type = 'bool'
        else:
            feature_type = 'str'
        return feature_type

## RemoveDuplicatesRows

In [ ]:

initial_rows = len(house_data)
duplicate_count = house_data.duplicated().sum()

print(f"Initial rows: {initial_rows}")
print(f"Duplicate rows: {duplicate_count}")
# ============================================================
# REMOVE DUPLICATES
# ============================================================
if duplicate_count > 0:

    house_data = (
        house_data
        .drop_duplicates()
        .reset_index(drop=True)
    )

    print(
        f"Removed {duplicate_count} duplicate rows."
    )

else:
    print("No duplicate rows found.")
# ============================================================
# FINAL RESULT
# ============================================================

print(
    f"Final rows: {len(house_data)}"
)
print(
    f"Final shape: {house_data.shape}"
)

## uniqueValues

In [ ]:
# ============================================================
# FIND COLUMNS WITH UNIQUE VALUES FOR EVERY ROW
# ============================================================

unique_columns = []

total_rows = len(house_data)

for column in house_data.columns:

    unique_values = house_data[column].nunique(
        dropna=False
    )



    if unique_values == total_rows:
        
        feature_type = detectValueType(column)

        unique_columns.append({'column_name':column, 'type':feature_type})


# ============================================================
# RESULT
# ============================================================
dfUniqueColumns = pd.DataFrame(unique_columns)
print("Columns where every value is unique:")
house_data = house_data.drop(columns = set(dfUniqueColumns['column_name'])) 
print(unique_columns)

print(
    "\nNumber of unique columns to delete:",
    len(unique_columns)
)


dfUniqueColumns['status'] = 'Deleted'
dfUniqueColumns['reason'] = 'Each row of the variable has an unique value'
dfDeletedColumns = pd.concat([dfDeletedColumns,dfUniqueColumns], ignore_index=False)

## Cardinality

In [ ]:
# we get all the features, because we want to confirm if we have not received a column with a default value
cat_cols = house_data.select_dtypes(include=['object', 'category','number']).columns
unique_cardinality_columns = []

# Show unique counts for categorical columns
print("\n RUNNING UNIQUE VALUES FOR CATEGORICAL COLUMNS")
print("=" * 60)
for col in cat_cols:
    n_unique = house_data[col].nunique()
    n_value = house_data[col][0]
    if n_unique== 1:
        print("=" * 60)
      
        print(f"{col:30} has the same value in all the rows   {n_value}")
        feature_type = detectValueType(column)
        unique_cardinality_columns.append({"column_name":col, 'type':feature_type})
        print("=" * 60)

    else:
        n_null = house_data[col].isnull().sum()
        # print(f"{col:30} {n_unique:4} unique values  (Nulls: {n_null})")
        

dfUniqueCardinalityColumns = pd.DataFrame(unique_cardinality_columns)

if len(dfUniqueCardinalityColumns)>0:
    house_data = house_data.drop(columns = set(dfUniqueCardinalityColumns['column_name'])) 

    dfUniqueCardinalityColumns['status'] = 'Deleted'
    dfUniqueCardinalityColumns['reason'] = 'All the rows has the name value'

    dfDeletedColumns = pd.concat([dfDeletedColumns,dfUniqueCardinalityColumns], ignore_index=False)
    print(
    tabulate(
            dfUniqueCardinalityColumns,
            headers="keys",
            tablefmt="pretty",
            showindex=False
        )
    )


else:
    print("\n THERE IS NOT UNIQUE VALUES FOR CATEGORICAL COLUMNS")


<!-- ## missingValuesColumn -->

In [ ]:
# # ============================================================
# # PARAMETERS
# # ============================================================

# MISSING_THRESHOLD = 50  # percentage


# # ============================================================
# # COLUMN-BY-COLUMN DATA PROFILE
# # ============================================================

# summary = []

# total_rows = len(house_data)

# for column in house_data.columns:

#     missing_count = house_data[column].isna().sum()

#     missing_percentage = (
#         missing_count / total_rows
#     ) * 100

#     unique_values = (
#         house_data[column].nunique(
#             dropna=True
#         )
#     )

#     data_type = (
#         house_data[column].dtype
#     )

#     summary.append({
#         "column": column,
#         "data_type": str(data_type),
#         "total_rows": total_rows,
#         "missing_count": missing_count,
#         "missing_percentage": missing_percentage,
#         "unique_values": unique_values
#     })


# # ============================================================
# # CREATE SUMMARY DATAFRAME
# # ============================================================

# column_summary = pd.DataFrame(summary)


# # ============================================================
# # COLUMNS TO DELETE
# # ============================================================

# columns_to_delete = column_summary.loc[
#     column_summary["missing_percentage"] > MISSING_THRESHOLD,
#     "column"
# ].tolist()


# print(
#     f"Missing-value threshold: {MISSING_THRESHOLD}%"
# )

# print(
#     f"Columns to delete: {len(columns_to_delete)}"
# )

# print(
#     "\nColumns deleted:"
# )

# print(columns_to_delete)


# # ============================================================
# # DELETE COLUMNS
# # ============================================================

# # house_data = house_data.drop(
# #     columns=columns_to_delete
# # )


# # ============================================================
# # FINAL SUMMARY
# # ============================================================

# print(
#     "\nOriginal number of columns:",
#     len(column_summary)
# )

# print(
#     "Final number of columns:",
#     house_data.shape[1]
# )

In [ ]:
ThresHoldNaValues

## FillingNAValuesChoosingBest

In [ ]:
# In these steps we will be fill the missing data of the variable where we are assuming that "the missin value" is not applicable
null_percentages = house_data.isnull().mean() * 100
high_null_cols = null_percentages[null_percentages > ThresHoldNaValues].index

for col in high_null_cols:
    if pd.api.types.is_numeric_dtype(house_data[col]):
        house_data[col] = house_data[col].fillna(0)
        valueType = detectValueType(col)
        dfSummaryFillNaValues.append({'feature_name':col, 'Type':valueType, 'Fill_with':'0','Reason':'Feature allows NA'})
        print(f"Filled {col} (numeric) with 0")
    else:
        house_data[col] = house_data[col].fillna('unknown')
        print(f"Filled {col} (categorical) with 'unknown'")
        valueType = detectValueType(col)
        dfSummaryFillNaValues.append({'feature_name':col, 'Type':valueType, 'Fill_with':'unknown','Reason':'Feature allows NA'})

## Collinearity

In [ ]:
import pandas as pd
import numpy as np

# ============================================================
# PARAMETERS
# ============================================================


TARGET = "SalePrice"


# ============================================================
# SEPARATE FEATURES
# ============================================================

X = house_data.drop(
    columns=[TARGET],
    errors="ignore"
)

numeric_cols = X.select_dtypes(
    include="number"
).columns.tolist()

categorical_cols = X.select_dtypes(
    exclude="number"
).columns.tolist()


# ============================================================
# ONE-HOT ENCODE CATEGORICAL VARIABLES
# ============================================================

X_encoded = pd.get_dummies(
    X,
    columns=categorical_cols,
    drop_first=True,
    dtype=float
)


# ============================================================
# CORRELATION MATRIX
# ============================================================

corr_matrix = X_encoded.corr().abs()


# ============================================================
# CORRELATION WITH TARGET
# ============================================================

target_encoded = pd.concat(
    [
        X_encoded,
        house_data[TARGET]
    ],
    axis=1
)

target_corr = (
    target_encoded
    .corr()[TARGET]
    .drop(TARGET)
    .abs()
)


# ============================================================
# FIND HIGHLY CORRELATED ENCODED FEATURES
# ============================================================

# This part of the code delete the duplicate correlations

upper = corr_matrix.where(
    np.triu(
        np.ones(
            corr_matrix.shape
        ),
        k=1
    ).astype(bool)
)


# ============================================================
# MAP DUMMY → ORIGINAL COLUMN
# ============================================================

def get_original_column(dummy_column):

    # Numerical column
    if dummy_column in numeric_cols:
        return dummy_column

    # Categorical column
    for column in categorical_cols:

        prefix = column + "_"

        if dummy_column.startswith(prefix):
            return column

    return dummy_column


# ============================================================
# COMPARE CORRELATED FEATURES
# ============================================================

pairs = []

for col1 in upper.columns:

    for col2 in upper.index:

        correlation = upper.loc[
            col2,
            col1
        ]

        if pd.isna(correlation):
            continue

        if correlation >= (ThresholdCollinearity/100):

            original_col1 = get_original_column(
                col1
            )

            original_col2 = get_original_column(
                col2
            )

            # Don't compare categories belonging
            # to the same original column

            if original_col1 == original_col2:
                continue


            # Target correlation of each encoded variable

            target_corr_1 = target_corr[col1]
            target_corr_2 = target_corr[col2]


            # =================================================
            # DECIDE WHICH ORIGINAL COLUMN TO REMOVE
            # =================================================

            if target_corr_1 < target_corr_2:

                deleted = original_col1
                kept = original_col2

            else:

                deleted = original_col2
                kept = original_col1


            pairs.append({

                "original_column_1":
                    original_col1,

                "encoded_column_1":
                    col1,

                "target_corr_1":
                    round(
                        target_corr_1,
                        4
                    ),

                "original_column_2":
                    original_col2,

                "encoded_column_2":
                    col2,

                "target_corr_2":
                    round(
                        target_corr_2,
                        4
                    ),

                "multicollinearity":
                    round(
                        correlation,
                        4
                    ),

                "KEEP":
                    kept,

                "DELETE":
                    deleted

            })


# ============================================================
# REPORT
# ============================================================

report = pd.DataFrame(pairs)




# ============================================================
# DELETE ORIGINAL COLUMNS
# ============================================================





# ============================================================
# SUMMARY
# ============================================================

print("\n==============================")
print("SUMMARY")
print("==============================")

print(
    "Threshold:",
    ThresholdCollinearity 
)





reportAdjustmentCollinearity = report.drop(columns = {'encoded_column_1','encoded_column_2','KEEP','DELETE'})

reportAdjustmentCollinearity = reportAdjustmentCollinearity.groupby(['original_column_1','original_column_2'])[['target_corr_1','target_corr_2','multicollinearity']].mean().reset_index()

reportAdjustmentCollinearity= reportAdjustmentCollinearity[['original_column_1','target_corr_1', 'original_column_2', 
       'target_corr_2', 'multicollinearity']]
reportAdjustmentCollinearity['DELETE'] = np.where(reportAdjustmentCollinearity['target_corr_1']>reportAdjustmentCollinearity['target_corr_2'],  reportAdjustmentCollinearity['original_column_2'],
reportAdjustmentCollinearity['original_column_1'] )

duplicatedFeatures = reportAdjustmentCollinearity[reportAdjustmentCollinearity['DELETE'].duplicated()]

if len(duplicatedFeatures)>0:
    raise Exception('ERRROR DUPLICATES FEATURES')



dfDeletedColumnCollinearity = []
for feature in reportAdjustmentCollinearity['DELETE']:
    
    feature_type = detectValueType(feature)
    dfDeletedColumnCollinearity.append({'column_name':feature, 'type':feature_type} )

dfDeletedColumnCollinearity = pd.DataFrame(dfDeletedColumnCollinearity)
dfDeletedColumnCollinearity['status'] = 'Deleted'
dfDeletedColumnCollinearity['reason'] = 'Collinearity'



if not report.empty:

    print(
        tabulate(
            reportAdjustmentCollinearity,
            headers="keys",
            tablefmt="pretty",
            showindex=False
        )
    )


else:

    columns_to_delete = []


house_data = house_data.drop(
    columns=list(reportAdjustmentCollinearity['DELETE']),
    errors="ignore"
)


print(
    "Columns deleted:",
    len(reportAdjustmentCollinearity)
)


print(
    "Remaining columns:",
    house_data.shape[1]
)
dfDeletedColumns = pd.concat([dfDeletedColumns,dfDeletedColumnCollinearity])

## Correlation

In [ ]:
# ============================================================
# PARAMETERS
# ============================================================



TARGET = "SalePrice"


# ============================================================
# SEPARATE NUMERICAL AND CATEGORICAL COLUMNS
# ============================================================

numeric_columns = house_data.select_dtypes(
    include="number"
).columns.tolist()

categorical_columns = house_data.select_dtypes(
    exclude="number"
).columns.tolist()


# Remove target from numerical features

numeric_columns.remove(TARGET)


# ============================================================
# NUMERICAL CORRELATION
# ============================================================

numeric_correlation = (
    house_data[numeric_columns + [TARGET]]
    .corr()[TARGET]
    .drop(TARGET)
)


numeric_summary = pd.DataFrame({

    "column": numeric_correlation.index,

    "correlation":
        numeric_correlation.values,

    "absolute_correlation":
        numeric_correlation.abs().values,

    "column_type":
        "numerical"

})


# ============================================================
# CATEGORICAL → DUMMY VARIABLES
# ============================================================

categorical_data = pd.get_dummies(
    house_data[categorical_columns],
    dummy_na=True,
    dtype=int
)


# ============================================================
# CORRELATION OF EACH DUMMY WITH SALEPRICE
# ============================================================

categorical_correlations = (
    categorical_data
    .corrwith(
        house_data[TARGET]
    )
)


# ============================================================
# REJOIN DUMMY CORRELATIONS TO ORIGINAL COLUMN
# ============================================================

categorical_summary = []

for column in categorical_columns:

    # Get all dummy columns belonging
    # to this original categorical column

    dummy_columns = [
        dummy
        for dummy in categorical_data.columns
        if dummy.startswith(column + "_")
    ]

    # Correlations for this original column

    correlations = (
        categorical_correlations[
            dummy_columns
        ]
    )

    # Find strongest relationship

    strongest_correlation = (
        correlations
        .loc[
            correlations.abs().idxmax()
        ]
    )

    strongest_category = (
        correlations
        .abs()
        .idxmax()
    )

    categorical_summary.append({

        "column": column,

        "correlation":
            strongest_correlation,

        "absolute_correlation":
            abs(strongest_correlation),

        "strongest_category":
            strongest_category,

        "column_type":
            "categorical"

    })


categorical_summary = pd.DataFrame(
    categorical_summary
)


# ============================================================
# COMBINE NUMERICAL + CATEGORICAL
# ============================================================

correlation_summary = pd.concat(
    [
        numeric_summary,
        categorical_summary
    ],
    ignore_index=True
)


# ============================================================
# SORT
# ============================================================

correlation_summary = (
    correlation_summary
    .sort_values(
        "absolute_correlation",
        ascending=False
    )
    .reset_index(drop=True)
)


# ============================================================
# APPLY THRESHOLD
# ============================================================

correlation_summary["selected"] = (
    correlation_summary[
        "absolute_correlation"
    ]
    >= (ThresholdCorrelation/100)
)


# ============================================================
# DISPLAY
# ============================================================


print(
    tabulate(
        correlation_summary,
        headers="keys",
        tablefmt="pretty",
        showindex=False
    )
)




dfDeletedColumnsCorrelation = []
listDropColumns = correlation_summary[correlation_summary['selected'] == False]['column']


for feature in list(listDropColumns):
    
    feature_type = detectValueType(feature)
    dfDeletedColumnsCorrelation.append({'column_name':feature, 'type':feature_type} )

dfDeletedColumnsCorrelation = pd.DataFrame(dfDeletedColumnsCorrelation)
dfDeletedColumnsCorrelation['status'] = 'Deleted'
dfDeletedColumnsCorrelation['reason'] = 'Correlation'


dfDeletedColumns = pd.concat([dfDeletedColumns,dfDeletedColumnsCorrelation])



house_data.drop(columns=list(listDropColumns), inplace=True)



## NullRowValues

In [ ]:
# ============================================================
# NULL ROW THRESHOLD
# ============================================================
NULL_THRESHOLD = 20  # percentage
# ============================================================
# CALCULATE NULL PERCENTAGE FOR EACH ROW
# ============================================================

row_null_percentage = (
    house_data.isna().mean(axis=1) * 100
)


# ============================================================
# IDENTIFY ROWS TO DELETE
# ============================================================

rows_to_delete = (
    row_null_percentage > NULL_THRESHOLD
)


print(
    f"Null threshold: {NULL_THRESHOLD}%"
)

print(
    f"Rows to delete: {rows_to_delete.sum()}"
)


# ============================================================
# DELETE ROWS
# ============================================================

house_data = house_data.loc[
    ~rows_to_delete
].reset_index(drop=True)


# ============================================================
# RESULT
# ============================================================

print(
    f"Remaining rows: {len(house_data)}"
)

## FillingNAValues

In [ ]:
import pandas as pd
import numpy as np
import tkinter as tk
from tkinter import messagebox

# Calculate missing percentages
null_percentages = house_data.isnull().mean() * 100

# Separate columns by missing percentage
high_missing_cols = null_percentages[null_percentages > thresholdMissingValues].index 
low_missing_cols = null_percentages[(null_percentages > 0) & (null_percentages <= thresholdMissingValues)].index  # 0-20% missing

# Handle columns with >20% missing values (pop-up confirmation)
if len(high_missing_cols) > 0:
    # Create pop-up message
    root = tk.Tk()
    root.withdraw()  # Hide the main window
    
    cols_list = '\n'.join(high_missing_cols)
    message = f"The following columns have >20% missing values:\n\n{cols_list}\n\nDo you want to continue and fill them?"
    
    response = messagebox.askyesno("Missing Values Alert", message)
    root.destroy()
    
    if response:  # If user clicks "Yes"
        print("Filling columns with >20% missing values...")
        for col in high_missing_cols:
            if pd.api.types.is_numeric_dtype(house_data[col]):
                house_data[col] = house_data[col].fillna(0)
                print(f"  - {col}: Filled with 0 (numeric)")
            else:
                house_data[col] = house_data[col].fillna('unknown')
                print(f"  - {col}: Filled with 'unknown' (categorical)")
    else:  # If user clicks "No"
        print("Operation cancelled. No columns were filled.")
        # Optionally, you can stop execution here
        # raise SystemExit("User chose to stop")

# Handle columns with <=20% missing values (automatic fill)
if len(low_missing_cols) > 0:
    print("\nFilling columns with <=20% missing values...")
    for col in low_missing_cols:
        if pd.api.types.is_numeric_dtype(house_data[col]):
            # Fill with median
            median_val = house_data[col].median()
            valueType = detectValueType(col)
            dfSummaryFillNaValues.append({'feature_name':col, 'Type':valueType, 'Fill_with':f'{median_val:.2f}','Reason':'Missing values'})
            house_data[col] = house_data[col].fillna(median_val)
            print(f"  - {col}: Filled with median ({median_val:.2f})")
        else:
            # Fill with most frequent category (mode)
            mode_val = house_data[col].mode()[0] if not house_data[col].mode().empty else 'unknown'
            dfSummaryFillNaValues.append({'feature_name':col, 'Type':valueType, 'Fill_with':f"{mode_val}",'Reason':'Missing values'})

            house_data[col] = house_data[col].fillna(mode_val)
            print(f"  - {col}: Filled with mode ('{mode_val}')")
            
# Verify no nulls remain
remaining_nulls = house_data.isnull().sum()
if remaining_nulls.sum() > 0:
    print(remaining_nulls[remaining_nulls > 0])
    raise Exception(  print("\n⚠️ Warning: Some null values still remain:"))
else:
    print("\n✅ All null values have been filled!")

## SplitProcess

In [ ]:
from sklearn.model_selection import train_test_split

X = house_data.drop(columns= 'SalePrice')
y = house_data[ 'SalePrice']

from sklearn.model_selection import train_test_split
import pandas as pd

# Create 10 bins from the continuous target
y_bins = pd.qcut(
    y,
    q=10,
    labels=False,
    duplicates='drop'
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=(testSplitSize/100),
    random_state=42,
    stratify=y_bins
)

## dataStandarizacion

## combinationVariables

In [ ]:
# from itertools import combinations

# exog_vars = list(house_data.columns)

# result = []
# for r in range(23, len(exog_vars) + 1):
#     result.extend(combinations(exog_vars, r))

# result = [list(x) for x in result]


In [ ]:

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from category_encoders import TargetEncoder


def dataCategoryTransfor(CombinationVariables, data, y):
       

        # -----------------------------
        # 1. Separar X e y
        # -----------------------------

        X = data[CombinationVariables]
     

        # -----------------------------
        # 2. Identificar categóricas
        # -----------------------------

        categorical_cols = X.select_dtypes(
            include=['object', 'category']
        ).columns.tolist()

        # -----------------------------
        # 3. Separar según cardinalidad
        # -----------------------------

        onehot_cols = [
            col for col in categorical_cols
            if X[col].nunique() <= thresholdOneHotEncoding
        ]

        target_cols = [
            col for col in categorical_cols
            if X[col].nunique() > thresholdOneHotEncoding
        ]

        print("OneHot:", onehot_cols)
        print("Target Encoding:", target_cols)

        # -----------------------------
        # 4. Crear transformador
        # -----------------------------
        preprocessor = ColumnTransformer(
            transformers=[
                (
                    'onehot',
                    OneHotEncoder(
                        handle_unknown='ignore',
                        sparse_output=False
                    ),
                    onehot_cols
                ),
                (
                    'target',
                    TargetEncoder(),
                    target_cols
                )
            ],
            remainder='passthrough'
        )

        # -----------------------------
        # 5. Transformar
        # -----------------------------

        X_encoded = preprocessor.fit_transform(X, y)

        # Obtener nombres de columnas
        feature_names = preprocessor.get_feature_names_out()

        xEncoded = pd.DataFrame(
            X_encoded,
            columns=feature_names,
            index=X.index
        )
        return xEncoded, preprocessor, onehot_cols, target_cols


In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import RobustScaler, StandardScaler


scalerRobust = RobustScaler()


scalerStandar = StandardScaler()

def calculate_outlier_percentage(data, multiplier=1.5):
    """
    Calculate percentage of outliers using IQR method
    
    Parameters:
    -----------
    data : pandas Series or array
    multiplier : float, default=1.5
        IQR multiplier (Tukey's rule)
    
    Returns:
    --------
    float: Percentage of outliers
    """
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - multiplier * IQR
    upper_bound = Q3 + multiplier * IQR
    
    outliers = data[(data < lower_bound) | (data > upper_bound)]
    
    return (len(outliers) / len(data)) * 100


def decide_scaler_for_column(data, verbose=False):
    """
    Automatically decide which scaler to use for a column
    
    Parameters:
    -----------
    data : pandas Series
        The column data to analyze
    verbose : bool, default=True
        Print detailed analysis
    
    Returns:
    --------
    tuple: (recommendation, scores)
        recommendation: 'StandardScaler' or 'RobustScaler'
        scores: dict with scores for each scaler
    """
    # Calculate metrics
    n = len(data)
    outlier_pct = calculate_outlier_percentage(data)
    skewness = data.skew()
    mean = data.mean()
    median = data.median()
    std = data.std()
    kurtosis = data.kurtosis()
    
    # Decision scoring system
    scores = {'StandardScaler': 0, 'RobustScaler': 0}
    
    # Score for StandardScaler
    if outlier_pct < 2:
        scores['StandardScaler'] += 30
    if outlier_pct < 5:
        scores['StandardScaler'] += 15
    if abs(skewness) < 0.5:
        scores['StandardScaler'] += 25
    if abs(skewness) < 1:
        scores['StandardScaler'] += 10
    if abs(mean - median) < 0.1 * std:
        scores['StandardScaler'] += 20
    if n > 1000:
        scores['StandardScaler'] += 10
    
    # Score for RobustScaler
    if outlier_pct > 5:
        scores['RobustScaler'] += 30
    if outlier_pct > 10:
        scores['RobustScaler'] += 20
    if abs(skewness) > 1.5:
        scores['RobustScaler'] += 25
    if abs(skewness) > 2:
        scores['RobustScaler'] += 15
    if abs(mean - median) > 0.2 * std:
        scores['RobustScaler'] += 20
    if n < 500:
        scores['RobustScaler'] += 10
    
    # Decision
    if scores['RobustScaler'] > scores['StandardScaler']:
        recommendation = 'RobustScaler'
        confidence = scores['RobustScaler'] / (scores['RobustScaler'] + scores['StandardScaler'])
    else:
        recommendation = 'StandardScaler'
        confidence = scores['StandardScaler'] / (scores['RobustScaler'] + scores['StandardScaler'])
    
    if verbose:
        print(f"\n📊 SCALER DECISION")
        print("="*50)
        print(f"Outliers: {outlier_pct:.1f}%")
        print(f"Skewness: {skewness:.2f}")
        print(f"Mean/Median: {mean:.2f} / {median:.2f} (ratio: {mean/median:.2f})")
        print(f"Kurtosis: {kurtosis:.2f}")
        print(f"\nScores:")
        print(f"  StandardScaler: {scores['StandardScaler']}")
        print(f"  RobustScaler: {scores['RobustScaler']}")
        print(f"\n✅ RECOMMENDATION: {recommendation} (confidence: {confidence:.0%})")
        
        # Show reasoning
        if recommendation == 'RobustScaler':
            print(f"\n   Why RobustScaler?")
            if outlier_pct > 5:
                print(f"   • High outlier percentage ({outlier_pct:.1f}%)")
            if abs(skewness) > 1.5:
                print(f"   • High skewness ({skewness:.2f})")
            if abs(mean - median) > 0.2 * std:
                print(f"   • Mean ({mean:.2f}) far from median ({median:.2f})")
        else:
            print(f"\n   Why StandardScaler?")
            if outlier_pct < 2:
                print(f"   • Low outlier percentage ({outlier_pct:.1f}%)")
            if abs(skewness) < 0.5:
                print(f"   • Low skewness ({skewness:.2f})")
            if abs(mean - median) < 0.1 * std:
                print(f"   • Mean ({mean:.2f}) close to median ({median:.2f})")
    
    return recommendation, scores


# ============================================
# USAGE EXAMPLE
# ============================================


for col in X_train.select_dtypes(include=[np.number]).columns:
    data = X_train[col].dropna()
    print(f'Analyzing: {col}')
    if col == 'SalePrice':
        print('Target value')
        continue
    if data.nunique() > ThresHoldStandarizacion or data.between(0, 1).all() == True:
   
    #     print('*'*50)
        recommendation, scores = decide_scaler_for_column(data)
        
        if recommendation == 'RobustScaler':
                print(recommendation)
                X_train[[col]] = scalerRobust.fit_transform(X_train[[col]])
                X_test[[col]] = scalerRobust.fit_transform(X_test[[col]])

                dfSummaryDataStandarizacion.append({'Feature':col, 'type_standarization':recommendation, 'reason':'there are outliers' })

        if recommendation == 'StandardScaler':
                print(recommendation)
                X_train[[col]] = scalerStandar.fit_transform(X_train[[col]])
                X_test[[col]] = scalerStandar.fit_transform(X_test[[col]])
                dfSummaryDataStandarizacion.append({'Feature':col, 'type_standarization':recommendation, 'reason':'there are not outliers' })

        
    else :
        print(f'No applying any process to the numeric column because there are less than ThresHoldStandarizacion uniques values')
        dfSummaryDataStandarizacion.append({'Feature':col, 'type_standarization':'No applicable ', 'reason':'Low unique values or the feature is in range 0 to 1 ' })

dfSummaryDataStandarizacion = pd.DataFrame(dfSummaryDataStandarizacion)

## BestModelAndParameters

In [ ]:
from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso,
    ElasticNet,
    BayesianRidge
)

from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    RandomForestRegressor,
    ExtraTreesRegressor
)

from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
models = {
    # Linear
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(),
    "ElasticNet": ElasticNet(),
    "Bayesian Ridge": BayesianRidge(),

    # Tree / Ensemble
    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),

    "Extra Trees": ExtraTreesRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=10, random_state=42
    ),

    # Other
    "SVR": SVR(),
    "KNN": KNeighborsRegressor()
}

def crossValidation(xEncoded, y , CombinationVariables ):
 



    from sklearn.model_selection import cross_validate, KFold

    cv = KFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    scoring = {
        "R2": "r2",
        "MAE": "neg_mean_absolute_error",
        "RMSE": "neg_root_mean_squared_error"
    }

    results = []

    for name, model in models.items():

        scores = cross_validate(
            model,
            xEncoded,
            y,
            cv=cv,
            scoring=scoring
        )

        results.append({
            "Model": name,
            "NVariables":len(CombinationVariables),
            "R2": scores["test_R2"].mean(),
            "MAE": -scores["test_MAE"].mean(),
            "RMSE": -scores["test_RMSE"].mean(),
            "Variables":CombinationVariables.to_json()
        })

    results_df = pd.DataFrame(results)

    results_df = results_df.sort_values("RMSE").reset_index(drop=True)
    results_df['rank'] = results_df.index + 1 
    return results_df


In [ ]:
selectedVariables = len(correlation_summary[correlation_summary['selected'] == True])
results_df_iterarion = []
for i in range(selectedVariables-iteration_offset,selectedVariables+1):
        CombinationVariables = correlation_summary[correlation_summary['selected'] == True].sort_values(by='absolute_correlation', ascending=False)['column'][:i]

        
        X_train_cross_validation, preprocessor,  onehot_cols, target_cols = dataCategoryTransfor(CombinationVariables, X_train, y_train)
        
        results_df =  crossValidation(X_train_cross_validation, y_train , CombinationVariables )
    
        copy = results_df
        if len(results_df_iterarion) == 0:
            results_df_iterarion = results_df.copy()
        else:
            results_df_iterarion = pd.concat([results_df_iterarion,results_df])
            


In [ ]:
bestModelName, bestFeatures, R2Value,MAEValue,RMSEValue = results_df_iterarion.sort_values(by='MAE', ascending = True).reset_index(drop=True)[['Model','Variables','R2','MAE','RMSE']].iloc[0]
import ast

dictionary = ast.literal_eval(bestFeatures)
dictionary = list(dictionary.values())
X_train_randomSearch, preprocessor,  onehot_cols, target_cols = dataCategoryTransfor(dictionary, X_train, y_train)
X_test_randomSearch = preprocessor.transform(X_test)

dfEncodingSummary = pd.DataFrame([onehot_cols,target_cols]).transpose()
dfEncodingSummary.columns = ['onehotEncoding','targetEncoding']

In [ ]:

dfBestModelCombinanation = {'ModelChosen':bestModelName,'NumberBestCombinationFeatures':len(dictionary),
                          
 'R2Value':f'{R2Value:.2f}'
                             ,'MAEValue':f'{MAEValue:.0f}','RMSE':   f'{RMSEValue:.0f}',    'Description':'Evaluation combination see sheet FeatureCombIteration_summary'}

dfBestModelCombinanation = pd.DataFrame.from_dict(dfBestModelCombinanation, orient= 'index').transpose()


,ModelChosen,NumberBestCombinationFeatures,Description,R2Value,MAEValue,RMSE
0,Extra Trees,56,Evaluation combination see sheet FeatureCombIt...,0.86,17078,28931


In [ ]:
import json

dfBestFeatures = json.loads(bestFeatures)

dfBestFeatures = pd.DataFrame(dfBestFeatures.values(), columns= ['Best_features'])


In [ ]:
from sklearn.model_selection import RandomizedSearchCV, KFold
from scipy.stats import uniform, loguniform, randint

from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso,
    ElasticNet,
    BayesianRidge
)

from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor
)

from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor


# --------------------------------------------------
# Hyperparameter distributions
# --------------------------------------------------

param_distributions = {

    "Linear Regression": {},

    "Ridge": {
        "alpha": loguniform(1e-3, 1000)
    },

    "Lasso": {
        "alpha": loguniform(1e-5, 100)
    },

    "ElasticNet": {
        "alpha": loguniform(1e-5, 100),
        "l1_ratio": uniform(0, 1)
    },

    "Bayesian Ridge": {
        "alpha_1": loguniform(1e-7, 1e-2),
        "alpha_2": loguniform(1e-7, 1e-2),
        "lambda_1": loguniform(1e-7, 1e-2),
        "lambda_2": loguniform(1e-7, 1e-2)
    },

    "Random Forest": {
        "n_estimators": randint(100, 500),
        "max_depth": [None, 10, 20, 30, 40],
        "min_samples_split": randint(2, 20),
        "min_samples_leaf": randint(1, 10),
        "max_features": ["sqrt", "log2", 1.0]
    },

    "Extra Trees": {
        "n_estimators": randint(100, 500),
        "max_depth": [None, 10, 20, 30, 40],
        "min_samples_split": randint(2, 20),
        "min_samples_leaf": randint(1, 10),
        "max_features": ["sqrt", "log2", 1.0]
    },

    "Gradient Boosting": {
        "n_estimators": randint(100, 500),
        "learning_rate": loguniform(0.01, 0.3),
        "max_depth": randint(2, 8),
        "min_samples_split": randint(2, 20),
        "min_samples_leaf": randint(1, 10),
        "subsample": uniform(0.6, 0.4)
    },

    "SVR": {
        "C": loguniform(1e-2, 1000),
        "epsilon": loguniform(1e-4, 1),
        "gamma": ["scale", "auto"]
    },

    "KNN": {
        "n_neighbors": randint(2, 30),
        "weights": ["uniform", "distance"],
        "p": [1, 2]
    }
}

## RandomizedSearchCV

In [88]:
best_models = {}
results = []

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for name, model in models.items():

    if name == bestModelName:
        print(f"\nOptimizing: {name}")
        search = RandomizedSearchCV(
            estimator=model,
            param_distributions=param_distributions[name],
            n_iter=30,
            scoring="neg_root_mean_squared_error",
            cv=cv,
            random_state=42,
            n_jobs=-1,
            verbose=1
        )

        search.fit(X_train_randomSearch, y_train)

        best_models[name] = search.best_estimator_

        results.append({
            "Model": name,
            "Best RMSE": -search.best_score_,
            "Best Parameters": search.best_params_
        })


Optimizing: Extra Trees
Fitting 5 folds for each of 30 candidates, totalling 150 fits


In [91]:
resultsRandomSearchCV = pd.DataFrame(results)

## FinalModelPrediction

In [102]:
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score


BestParameters = results[0]['Best Parameters']
BestModel = models[bestModelName]

BestModel.set_params(**BestParameters)
BestModel.fit(X_train_randomSearch, y_train)


y_test_predicted = BestModel.predict(X_test_randomSearch)

mse = mean_squared_error(y_test, y_test_predicted)
rmse = mean_squared_error(y_test, y_test_predicted) ** 0.5

r2 = r2_score(y_test, y_test_predicted)

c:\Users\herma\anaconda3\envs\cursoml\lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but ExtraTreesRegressor was fitted with feature names
  warnings.warn(


In [ ]:
df_metrics_final = pd.DataFrame({
    "Metric": ["MSE", "RMSE", "R2"],
    "Value": [mse, rmse, r2]
})



,Metric,Value
0,MSE,7.336880e+08
1,RMSE,2.708668e+04
2,R2,8.855554e-01


## exportResume

In [108]:
from openpyxl.styles import Font, Alignment

dfSummaryFillNaValues = pd.DataFrame(dfSummaryFillNaValues)


dfParametersExport = pd.DataFrame.from_dict(
    dictParameters,
    orient="index",
    columns=["value","description"]

).reset_index()
dfParametersExport.columns = ['parameter', 'value', 'description']


with pd.ExcelWriter("model_results.xlsx", engine="openpyxl", ) as writer:

    dfParametersExport.to_excel(
        writer,
        sheet_name="Summary",
        startrow=1,
        startcol=1,
        index=False
    )


    ws = writer.book["Summary"]
 
    ws.cell(row=1, column=2).value = "1.) Parameters"
    ws.merge_cells(
        start_row=1,
        start_column=2,
        end_row=1,
        end_column=2
    )
    ws.cell(row=1, column=2).font = Font(bold=True, size=14)
    ws.cell(row=1, column=2).alignment = Alignment(horizontal="center")

    ws.cell(row=14, column=2).value = "2.) Summary deleted features"
    ws.merge_cells(
        start_row=14,
        start_column=2,
        end_row=14,
        end_column=2
    )
    ws.cell(row=14, column=2).font = Font(bold=True, size=14)
    ws.cell(row=14, column=2).alignment = Alignment(horizontal="center")



    dfDeletedColumns['reason'].value_counts().reset_index().to_excel(
        writer,
        sheet_name="Summary",
        startrow=15,
        startcol=1,
        index=False
    )



    # start for RandomSearchCV

    ws.cell(row=21, column=2).value = "4.) RandomSearchCV Results"
    ws.merge_cells(
        start_row=21,
        start_column=2,
        end_row=21,
        end_column=2
    )
    ws.cell(row=21, column=2).font = Font(bold=True, size=14)
    ws.cell(row=21, column=2).alignment = Alignment(horizontal="center")


    resultsRandomSearchCV.to_excel(
            writer,
            sheet_name="Summary",
            startrow=21,
            startcol=1,
            index=False
        )
    
    



# start for dfBestModelCombinanation
    dfBestModelCombinanation.to_excel(
        writer,
        sheet_name="Summary",
        startrow=15,
        startcol=5,
        index=False
    )


    ws.cell(row=15, column=7).value = "3.) Best features combination and model"
    ws.merge_cells(
        start_row=15,
        start_column=15,
        end_row=15,
        end_column=15
    )
    ws.cell(row=15, column=7).font = Font(bold=True, size=14)
    ws.cell(row=15, column=7).alignment = Alignment(horizontal="center")

# end  for dfBestModelCombinanation


# start  results with training data

    df_metrics_final.to_excel(
            writer,
            sheet_name="Summary",
            startrow=21,
            startcol=5,
            index=False
        )
    
    ws.cell(row=21, column=7).value = "5.) testing data evaluation"
    ws.merge_cells(
        start_row=21,
        start_column=15,
        end_row=21,
        end_column=15
    )
    ws.cell(row=21, column=7).font = Font(bold=True, size=14)
    ws.cell(row=21, column=7).alignment = Alignment(horizontal="center")

# end  results with training data




    dfDeletedColumns.to_excel(
        writer,
        sheet_name="Deleted_columns",
        startrow=1,
        startcol=1,
        index=False
    )
    correlation_summary.to_excel(
            writer,
            sheet_name="Correlation_summary",
            startrow=1,
            startcol=1,
            index=False
        )
    dfBestFeatures.to_excel(
                writer,
                sheet_name="BestFeatures_list",
                startrow=1,
                startcol=1,
                index=False
            )


        
    reportAdjustmentCollinearity.to_excel(
                writer,
                sheet_name="Collinearity_summary",
                startrow=1,
                startcol=1,
                index=False
            )

    dfSummaryFillNaValues.to_excel(
                    writer,
                    sheet_name="FillNa_Summary",
                    startrow=1,
                    startcol=1,
                    index=False
                )

    dfSummaryDataStandarizacion.to_excel(
                        writer,
                        sheet_name="Standarizacion_summary",
                        startrow=1,
                        startcol=1,
                        index=False
                    )


    results_df_iterarion.to_excel(
                        writer,
                        sheet_name="FeatureCombIteration_summary",
                        startrow=1,
                        startcol=1,
                        index=False
                    )                
        
    dfEncodingSummary.to_excel(
                    writer,
                    sheet_name="FeatureCombIteration_summary",
                    startrow=1,
                    startcol=1,
                    index=False
                )                
    

    



    